# ScanNet HHA Forensic Diagnostics

Investigates the ~10pt val-MCA regression after switching from raw depth to HHA. Each section targets a specific hypothesis from the audit; run all to get the full picture, or skip to a specific section to test one hypothesis.

## Sections
1. **Setup** — mount Drive, stream-extract dataset to /dev/shm, clone repo for `compute_hha`/`apply_hha_aug`
2. **Norm-stats summary** — published values in `norm_stats.json` (mean/std/percentiles/nan_rate per channel)
3. **Empirical NaN rate** — compute from raw `_hha.pt` tensors (truth, not metadata)
4. **Floor-angle invariant** — hard assertion that floor pixels read ~180° (catches axis-inversion bugs)
5. **Resize aliasing (Hypothesis A)** — compare INTER_NEAREST vs mask-aware blur on a recomputed-at-native HHA
6. **Drop-list per-class impact (Hypothesis C)** — per-class scene-count loss from Phase 0 drops
7. **Extraction telemetry (Hypothesis D)** — pose-drop / blur-drop / decode-fail fraction from `processing_log.jsonl`
8. **Combined NaN at train-time (Hypothesis E)** — run `apply_hha_aug` and measure NaN before per-channel-mean replacement
9. **Distribution vs SUN HHA** — relative divergence per channel; histogram side-by-side
10. **Pass 2 frame correlation** — inter-frame distance + RGB similarity between Pass 1 and Pass 2 frames in rare classes
11. **Float16 round-trip sanity** — recompute mean/std from on-disk float16 tensors; compare to `norm_stats.json`
12. **Sample visualization** — 6 random samples with NaN overlay (sanity gut-check)

## 1. Setup

In [ ]:
# === Mount Drive + stream-extract dataset + clone repo ===
import os, subprocess, sys, json, glob, random, time, re
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.ndimage import label as cc_label  # connected-component count for §8
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

DRIVE_TAR  = '/content/drive/MyDrive/datasets/scannet_pretrain_256_hha.tar.gz'
# Auto-pick extraction target: /dev/shm if it has >=60 GB free, else /content
# (235 GB on Colab). /dev/shm sized varies by runtime (12-87 GB); the HHA
# dataset is ~30-50 GB extracted, so undersized /dev/shm silently truncates
# tar mid-extraction and leaves files like norm_stats.json zero-bytes.
import shutil as _shutil
_shm_free = _shutil.disk_usage('/dev/shm').free if os.path.isdir('/dev/shm') else 0
if _shm_free >= 60 * 1024**3:
    LOCAL_PATH = '/dev/shm/scannet_pretrain_256_hha'
    print(f'/dev/shm has {_shm_free/1024**3:.1f} GB free -> extracting there.')
else:
    LOCAL_PATH = '/content/scannet_pretrain_256_hha'
    print(f'/dev/shm only has {_shm_free/1024**3:.1f} GB free; extracting to /content/ instead.')
_extract_root = os.path.dirname(LOCAL_PATH)
REPO_PATH  = '/content/Multi-Stream-Neural-Networks'
REPO_URL   = 'https://github.com/clingergab/Multi-Stream-Neural-Networks.git'

print('=' * 60)
print('SETUP')
print('=' * 60)
if Path(LOCAL_PATH).exists() and any(Path(LOCAL_PATH, 'train').glob('*/*_rgb.pt')):
    print(f'Dataset already extracted: {LOCAL_PATH}')
elif Path(DRIVE_TAR).exists():
    print(f'Streaming {DRIVE_TAR} -> {_extract_root}/ ...')
    # pigz parallel decompress (~5-10x faster than single-threaded gunzip).
    # If pigz isn't pre-installed, the apt-get line installs it (~3s).
    !which pigz >/dev/null 2>&1 || apt-get install -y -qq pigz 2>&1 | tail -2
    # Use a real tar exit-code check via subprocess so we don't silently accept truncation.
    import subprocess as _sp
    _r = _sp.run(
        f'pigz -dc {DRIVE_TAR} | tar -xf - -C {_extract_root}/',
        shell=True, capture_output=True, text=True,
    )
    if _r.returncode != 0:
        # Show the first few real errors (filtered)
        for ln in _r.stderr.splitlines():
            if 'Ignoring unknown extended header' in ln: continue
            print(ln)
            if 'No space left on device' in ln:
                raise RuntimeError(
                    f'EXTRACTION TRUNCATED: {_extract_root} ran out of space. '
                    f'norm_stats.json and other late-written files are likely 0 bytes. '
                    f'Free up space or extract to a larger filesystem.'
                )
        raise RuntimeError(f'tar failed (rc={_r.returncode}); see stderr above.')
    print('Extracted.')
else:
    raise FileNotFoundError(f'No tarball at {DRIVE_TAR}')

if not Path(REPO_PATH).exists():
    !git clone --quiet {REPO_URL} {REPO_PATH}
else:
    !git -C {REPO_PATH} pull --quiet || echo 'WARN: git pull failed -- using current local state'
!git -C {REPO_PATH} rev-parse --short HEAD | xargs -I{{}} echo 'Repo HEAD: {{}}'
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from src.data_utils.hha import compute_hha
from src.data_utils.hha.augmentation import HHAAugConfig, apply_hha_aug

# Single-pass fs walk: counts + scene-set extraction for the leakage check below.
_train_scenes, _val_scenes = set(), set()
n_train = 0; n_val = 0
for _f in Path(LOCAL_PATH, 'train').rglob('*_rgb.pt'):
    n_train += 1
    _m = re.match(r'(scene\d+_\d+)_f', _f.name)
    if _m: _train_scenes.add(_m.group(1))
for _f in Path(LOCAL_PATH, 'val').rglob('*_rgb.pt'):
    n_val += 1
    _m = re.match(r'(scene\d+_\d+)_f', _f.name)
    if _m: _val_scenes.add(_m.group(1))
print(f'Train: {n_train:,} frames  Val: {n_val:,} frames')
!df -h {_extract_root} | tail -1

# Initialize forensic results dict (each section appends)
DIAG_RESULTS = {}

# Data-leakage scene-set check (uses sets built in the single fs walk above)
_overlap = _train_scenes & _val_scenes
if _overlap:
    print(f'\n!!! DATA LEAKAGE: {len(_overlap)} scenes in BOTH train and val !!!')
    print(f'    Examples: {sorted(_overlap)[:5]}')
    DIAG_RESULTS['data_leakage'] = {'verdict': 'FAIL', 'overlap_count': len(_overlap)}
else:
    print(f'\nData leakage check: PASS ({len(_train_scenes)} train scenes, {len(_val_scenes)} val scenes, 0 overlap)')
    DIAG_RESULTS['data_leakage'] = {'verdict': 'PASS', 'overlap_count': 0}


## 2. Norm-stats summary

In [ ]:
_ns_path = os.path.join(LOCAL_PATH, 'norm_stats.json')
NS = None
_ns_reason = None
if not os.path.isfile(_ns_path):
    _ns_reason = f'{_ns_path} not found'
elif os.path.getsize(_ns_path) == 0:
    _ns_reason = f'{_ns_path} is 0 bytes (empty)'
else:
    try:
        with open(_ns_path) as f:
            NS = json.load(f)
    except json.JSONDecodeError as _je:
        _ns_reason = f'{_ns_path} is not valid JSON: {_je}'

if NS is None:
    print(f'WARN: {_ns_reason}.')
    print(f'  norm_stats.json is written by cell 49 of preprocess.')
    print(f'  Sections needing NS will skip; section 3 still works (live empirical).')
    DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
    DIAG_RESULTS['norm_stats'] = {'verdict': 'INCONCLUSIVE', 'reason': _ns_reason}
    globals()['DIAG_RESULTS'] = DIAG_RESULTS

if NS is not None:
    print(f"RGB mean: {NS['rgb_mean']}")
    print(f"RGB std:  {NS['rgb_std']}")
    print(f"\nHHA mean: {NS.get('hha_mean')}")
    print(f"HHA std:  {NS.get('hha_std')}")
    print()
    print(f"{'Channel':<22}{'min':>9}{'max':>9}{'p1':>9}{'p99':>9}{'p99.9':>9}{'nan_rate':>11}")
    print('-' * 78)
    labels = ['ch0 disparity (1/m)', 'ch1 height (m)', 'ch2 angle (deg)']
    DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
    DIAG_RESULTS['norm_stats'] = {'verdict': 'INFO', 'has_hha_ranges': 'hha_ranges' in NS}
    globals()['DIAG_RESULTS'] = DIAG_RESULTS
    if 'hha_ranges' in NS:
        for c, lab in enumerate(labels):
            r = NS['hha_ranges'][c]
            print(f"{lab:<22}{r['min']:>9.3f}{r['max']:>9.3f}{r['p1']:>9.3f}"
                  f"{r['p99']:>9.3f}{r['p99_9']:>9.3f}{r['nan_rate']:>11.4f}")
        max_nan = max(NS['hha_ranges'][c]['nan_rate'] for c in range(3))
        print(f"\nWorst-channel NaN rate: {max_nan*100:.2f}%")
        if max_nan > 0.15:
            print('  WARN: NaN rate >15%; mean-fill creates large constant patches after normalize.')
        elif max_nan > 0.08:
            print('  Moderate NaN rate; combined with aug may compound (see section 8).')
        else:
            print('  NaN rate looks low; not the dominant cause on its own.')
    else:
        print('hha_ranges missing -- re-run cell 49 of preprocess notebook.')


## 3. Empirical NaN rate (live from tensors)
Sanity-check that on-disk reality matches `norm_stats.json` (which may be stale).

In [ ]:
hha_files = sorted(glob.glob(os.path.join(LOCAL_PATH, 'train', '*', '*_hha.pt')))
rng = random.Random(42)
sample = rng.sample(hha_files, min(200, len(hha_files)))
n_finite = np.zeros(3, dtype=np.int64); n_total = np.zeros(3, dtype=np.int64)
for p in sample:
    h = torch.load(p, weights_only=True).float().numpy()
    for c in range(3):
        m = np.isfinite(h[c])
        n_finite[c] += int(m.sum()); n_total[c] += int(h[c].size)
print(f'Sampled {len(sample)} HHA tensors:')
for c, lab in enumerate(['disparity', 'height   ', 'angle    ']):
    nr = 1.0 - n_finite[c] / max(n_total[c], 1)
    flag = '  <-- HIGH' if nr > 0.15 else ('  <-- moderate' if nr > 0.08 else '')
    print(f'  ch{c} {lab}: nan_rate = {nr:.4f}  ({nr*100:.2f}%){flag}')

## 4. Floor-angle invariant (Hypothesis B)
If `SCANNET_AXIS_INVERTED` was set wrong in Phase 0, the angle channel is inverted across the entire dataset (floors read ~0° instead of ~180°). The validation gate did NOT assert this. Test it now.

In [ ]:
# Floor and ceiling selection by HEIGHT channel (gravity-aligned), not by
# image rows — works for any camera tilt. Floor = bottom 5% of valid Z_world,
# ceiling = top 5%. The angle channel medians on those subsets should hit
# 180 (floor) and 0 (ceiling) respectively if the convention is correct.
_rng4 = random.Random(42)
n_frames = 50
files = _rng4.sample(hha_files, min(n_frames, len(hha_files)))
floor_angles = []
ceiling_angles = []
for p in files:
    h = torch.load(p, weights_only=True).float().numpy()  # [3, H, W]
    height = h[1]
    angle  = h[2]
    valid = np.isfinite(height) & np.isfinite(angle)
    if valid.sum() < 100:
        continue
    h_valid = height[valid]
    a_valid = angle[valid]
    # Floor: bottom 5% by height
    f_thresh = np.percentile(h_valid, 5)
    f_mask = h_valid <= f_thresh
    if f_mask.sum() > 0:
        floor_angles.append(float(np.median(a_valid[f_mask])))
    # Ceiling: top 5% by height
    c_thresh = np.percentile(h_valid, 95)
    c_mask = h_valid >= c_thresh
    if c_mask.sum() > 0:
        ceiling_angles.append(float(np.median(a_valid[c_mask])))

floor_angles = np.array(floor_angles)
ceiling_angles = np.array(ceiling_angles)

print(f'Sampled {len(files)} frames; {len(floor_angles)} usable for floor, '
      f'{len(ceiling_angles)} for ceiling.')

# Length guards
if len(floor_angles) == 0 or len(ceiling_angles) == 0:
    print('ERROR: no usable frames -- HHA tensors all-NaN or height channel empty.')
    floor_med = ceiling_med = float('nan')
    verdict_4 = 'INCONCLUSIVE'
else:
    floor_med = float(np.median(floor_angles))
    ceiling_med = float(np.median(ceiling_angles))
    print(f'Floor   angle median (low-height pixels): {floor_med:.1f}deg  (expect ~180)')
    print(f'Ceiling angle median (high-height pixels): {ceiling_med:.1f}deg  (expect ~0)')

    floor_ok = floor_med >= 150
    ceiling_ok = ceiling_med <= 30
    if floor_ok and ceiling_ok:
        verdict_4 = 'PASS'
        print(f'  PASS: convention is correct (floor=180, ceiling=0).')
    elif floor_med <= 30 and ceiling_med >= 150:
        verdict_4 = 'FAIL'
        print(f'  FAIL: convention INVERTED. floor and ceiling are swapped.')
        print(f'  This single bug fully explains the val-MCA regression.')
    elif 75 <= floor_med <= 105 and 75 <= ceiling_med <= 105:
        verdict_4 = 'FAIL'
        print(f'  FAIL: both reading ~90deg. Normals may be unoriented or')
        print(f'  the orient-toward-camera step is broken.')
    else:
        verdict_4 = 'WARN'
        print(f'  WARN: ambiguous. floor_ok={floor_ok}, ceiling_ok={ceiling_ok}')

# Dump for the summary cell
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
DIAG_RESULTS['floor_angle'] = {
    'verdict': verdict_4,
    'floor_median_deg': floor_med,
    'ceiling_median_deg': ceiling_med,
    'n_floor_frames': len(floor_angles),
    'n_ceiling_frames': len(ceiling_angles),
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS

# When verdict is FAIL/WARN, dump the 4 worst-floor-angle frames to disk so
# the operator can visually verify the convention claim.
if verdict_4 in ('FAIL', 'WARN') and len(floor_angles) >= 4:
    # Worst = farthest from expected 180
    err = np.abs(floor_angles - 180.0)
    worst_idx = np.argsort(err)[-4:]
    fig_o, axes_o = plt.subplots(1, 4, figsize=(14, 4))
    for k, wi in enumerate(worst_idx):
        h = torch.load(files[wi], weights_only=True).float().numpy()
        norm = TwoSlopeNorm(vmin=0, vcenter=90, vmax=180)
        axes_o[k].imshow(np.nan_to_num(h[2], nan=90.0), cmap='RdBu_r', norm=norm)
        axes_o[k].set_title(f'floor_angle={floor_angles[wi]:.1f}deg'); axes_o[k].axis('off')
    fig_o.suptitle('§4 outliers: worst 4 floor-angle frames')
    out = '/dev/shm/floor_angle_outliers.png'
    fig_o.savefig(out, dpi=80, bbox_inches='tight'); plt.close(fig_o)
    print(f'  Saved worst frames: {out}')
    DIAG_RESULTS['floor_angle']['outliers_image'] = out

# Two histograms side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
if len(floor_angles) > 0:
    axes[0].hist(floor_angles, bins=18, edgecolor='black')
axes[0].axvline(180, color='green', linestyle='--', label='expected (180)')
axes[0].axvline(0,   color='red',   linestyle='--', label='inverted (0)')
axes[0].axvline(90,  color='gray',  linestyle=':',  label='wall (90)')
axes[0].set_xlabel('floor angle median (deg)'); axes[0].set_xlim(0, 180); axes[0].legend()
axes[0].set_title('Floor pixels (low-height)')
if len(ceiling_angles) > 0:
    axes[1].hist(ceiling_angles, bins=18, edgecolor='black')
axes[1].axvline(0,   color='green', linestyle='--', label='expected (0)')
axes[1].axvline(180, color='red',   linestyle='--', label='inverted (180)')
axes[1].axvline(90,  color='gray',  linestyle=':',  label='wall (90)')
axes[1].set_xlabel('ceiling angle median (deg)'); axes[1].set_xlim(0, 180); axes[1].legend()
axes[1].set_title('Ceiling pixels (high-height)')
plt.tight_layout(); plt.show(); plt.close(fig)

## 5. Resize aliasing — TV-removability ratio (Hypothesis A)
Within-dataset metric: ratio of TV before vs after mask-aware smoothing. Aliasing produces stair-step TV that smooths away (high ratio); real scene edges survive smoothing (low ratio). Comparing the RATIO across ScanNet and SUN factors out their different native resolutions and scene-complexity differences.

In [ ]:
# Within-dataset TV-removability ratio: how much of the TV survives mask-aware
# smoothing (= real scene structure) vs how much disappears (= aliasing artifact).
# A high ratio (TV_before / TV_after >> 1) means the channel is dominated by
# stair-step aliasing that smooths away. Low ratio means the TV is real edges.
# Comparing this RATIO across ScanNet and SUN factors out their different
# native resolutions and capture-style differences.
import torch.nn.functional as F
_rng5 = random.Random(42)

def channel_tv(h_np, channel):
    ch = h_np[channel]
    valid = np.isfinite(ch)
    dh = np.abs(ch[:, 1:] - ch[:, :-1]); valid_h = valid[:, 1:] & valid[:, :-1]
    dv = np.abs(ch[1:, :] - ch[:-1, :]); valid_v = valid[1:, :] & valid[:-1, :]
    n = valid_h.sum() + valid_v.sum()
    return float((dh[valid_h].sum() + dv[valid_v].sum()) / n) if n > 0 else float('nan')

# Pre-build the smoothing kernel once
_k = 5
_sigma = 1.0
_coords = torch.arange(_k, dtype=torch.float32) - (_k - 1) / 2
_g1d = torch.exp(-(_coords ** 2) / (2 * _sigma ** 2)); _g1d = _g1d / _g1d.sum()
_g2d = (_g1d[:, None] * _g1d[None, :]).view(1, 1, _k, _k).repeat(3, 1, 1, 1)
_pad = _k // 2

def mask_aware_blur(h_tensor):
    valid = ~torch.isnan(h_tensor)
    h_zerofilled = torch.where(valid, h_tensor, torch.zeros_like(h_tensor))
    valid_f = valid.float()
    bh = F.conv2d(h_zerofilled.unsqueeze(0), _g2d, padding=_pad, groups=3).squeeze(0)
    bm = F.conv2d(valid_f.unsqueeze(0),      _g2d, padding=_pad, groups=3).squeeze(0)
    return torch.where(bm > 1e-6, bh / bm.clamp(min=1e-6),
                       torch.full_like(h_tensor, float('nan')))

def tv_ratio_per_channel(load_fn, n_samples=200):
    """Compute (TV_before / TV_after) ratios for n_samples loaded via load_fn(idx)."""
    ratios = [[], [], []]
    for i in range(n_samples):
        h_t = load_fn(i)
        if h_t is None: continue
        h_smooth_t = mask_aware_blur(h_t)
        h_np = h_t.numpy(); h_smooth_np = h_smooth_t.numpy()
        for c in range(3):
            tb = channel_tv(h_np, c)
            ta = channel_tv(h_smooth_np, c)
            if np.isfinite(tb) and np.isfinite(ta) and ta > 1e-6:
                ratios[c].append(tb / ta)
    return [float(np.mean(r)) if r else float('nan') for r in ratios]

print('Computing within-dataset TV-removability ratios on 200 ScanNet HHA tensors...')
scannet_files = _rng5.sample(hha_files, min(200, len(hha_files)))
def _load_scannet(i):
    return torch.load(scannet_files[i], weights_only=True).float() if i < len(scannet_files) else None
scannet_ratios = tv_ratio_per_channel(_load_scannet, n_samples=len(scannet_files))

print(f"\n{'Channel':<22}{'ScanNet TV ratio':>20}")
print('-' * 42)
labels = ['ch0 disparity (1/m)', 'ch1 height (m)', 'ch2 angle (deg)']
for c, lab in enumerate(labels):
    print(f'{lab:<22}{scannet_ratios[c]:>20.3f}')

# SUN comparison
sun_paths = [
    '/content/sunrgbd_19_hha/train/hha_tensors.pt',
    '/content/Multi-Stream-Neural-Networks/data/sunrgbd_19_hha/train/hha_tensors.pt',
]
sun_path = next((p for p in sun_paths if os.path.isfile(p)), None)
verdict_5 = 'INCONCLUSIVE'
# Auto-extract SUN train hha_tensors.pt if not on disk (~3-4 GB extracted)
if sun_path is None:
    sun_tar = '/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz'
    if os.path.isfile(sun_tar):
        print(f'\nSUN HHA tensors not found locally; extracting train/hha_tensors.pt from {sun_tar}...')
        os.makedirs('/content/sunrgbd_19_hha', exist_ok=True)
        # No shell=True -> no escape issues. Use tar's built-in gzip (-z); pigz
        # acceleration isn't worth the shell-pipe complexity for one ~3 GB file.
        _r = subprocess.run(
            ['tar', '-xzf', sun_tar,
             '-C', '/content/sunrgbd_19_hha',
             '--wildcards', '*/train/hha_tensors.pt',
             '--strip-components=1'],
            capture_output=True, text=True,
        )
        if _r.returncode == 0:
            sun_path = next((p for p in sun_paths if os.path.isfile(p)), None)
            if sun_path:
                print(f'Extracted: {sun_path} ({os.path.getsize(sun_path)/1024**3:.2f} GB)')
            else:
                # Fall back: find the file wherever tar put it
                _f = subprocess.run(
                    ['find', '/content', '-maxdepth', '5',
                     '-name', 'hha_tensors.pt', '-type', 'f'],
                    capture_output=True, text=True,
                )
                for cand in _f.stdout.strip().split('\n'):
                    if cand and '/train/' in cand and os.path.isfile(cand):
                        sun_path = cand
                        print(f'Found at: {sun_path}')
                        break
        else:
            print(f'Extraction failed (rc={_r.returncode}); stderr: {_r.stderr[:500]}')

if sun_path is None:
    print('\nSUN HHA tensors still not found; skipping cross-dataset TV-ratio comparison.')
else:
    print(f'\nComputing same metric on SUN HHA ({sun_path})...')
    # Use mmap to avoid OOM on standard Colab (3-4 GB tensor file)
    _mmap_ok = False
    try:
        T = torch.load(sun_path, weights_only=True, mmap=True)
        _mmap_ok = True
    except (TypeError, RuntimeError) as _e:
        # Older PyTorch / non-zip-format save -> fall back to full RAM load (~3-4 GB).
        T = torch.load(sun_path, weights_only=True)
    print(f'  SUN tensor load: {"mmap" if _mmap_ok else "full-RAM (older format)"}')
    sun_idx = _rng5.sample(range(len(T)), min(200, len(T)))
    def _load_sun(i):
        return T[sun_idx[i]].float() if i < len(sun_idx) else None
    sun_ratios = tv_ratio_per_channel(_load_sun, n_samples=len(sun_idx))
    print(f"\n{'Channel':<22}{'ScanNet ratio':>16}{'SUN ratio':>14}{'rel diff':>12}")
    print('-' * 64)
    max_excess = 0.0
    for c, lab in enumerate(labels):
        sc = scannet_ratios[c]; su = sun_ratios[c]
        excess = (sc - su) / max(su, 1e-6)  # how much more removable TV ScanNet has
        max_excess = max(max_excess, excess)
        print(f'{lab:<22}{sc:>16.3f}{su:>14.3f}{excess*100:>11.1f}%')
    if max_excess > 0.30:
        verdict_5 = 'WARN'
        print(f'\n  WARN: ScanNet has {max_excess*100:.0f}% more "smoothable" TV than SUN.')
        print(f'  Consistent with INTER_NEAREST aliasing in preprocess.')
    elif max_excess > 0.10:
        verdict_5 = 'WARN'
        print(f'\n  Mild: ScanNet ratio {max_excess*100:.0f}% above SUN.')
    else:
        verdict_5 = 'PASS'
        print(f'\n  PASS: TV-removability comparable.')

DIAG_RESULTS['resize_aliasing'] = {
    'verdict': verdict_5,
    'scannet_tv_ratios': scannet_ratios,
}
# Free the SUN tensor file from memory (was 3-4 GB).
try:
    del T
except NameError:
    pass

## 6. Drop-list per-class impact (Hypothesis C)
Phase 0 hard-drops scenes (missing axisAlignment, etc.) before HHA extraction. Raw-depth path didn't apply this filter. If common classes lost more scenes than rare ones, accuracy on the head classes drops disproportionately.

In [ ]:
# Resolve drop-list path; if found, fetch each dropped scene's .txt to classify.
drop_list_candidates = [
    '/content/drive/MyDrive/datasets/scannet_drop_list.json',
    '/content/drive/MyDrive/datasets/scannet_phase0/drop_list.json',
    '/content/drive/MyDrive/scannet_drop_list.json',
]
drop_list_path = next((p for p in drop_list_candidates if os.path.isfile(p)), None)
verdict_6 = 'INCONCLUSIVE'
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})

if drop_list_path is None:
    print('Drop list not found at standard paths.')
    print('Adjust drop_list_candidates above to your actual path.')
    DIAG_RESULTS['drop_list'] = {'verdict': verdict_6, 'reason': 'drop list not found'}
else:
    with open(drop_list_path) as f:
        DL = json.load(f)
    print(f'Drop list: {drop_list_path}')
    print(f'  convention_verified: {DL.get("convention_verified")}')
    print(f'  axis_alignment_inverted: {DL.get("axis_alignment_inverted")}')
    drops = DL.get('scenes', {})
    print(f'  Total dropped scenes: {len(drops)}')
    reason_counts = Counter()
    for sid, reason in drops.items():
        key = reason if isinstance(reason, str) else type(reason).__name__
        reason_counts[key.split(':')[0] if ':' in key else key] += 1
    print('  Reason histogram:')
    for r, c in reason_counts.most_common():
        print(f'    {r:35s}: {c}')

    # Re-download .txt for each dropped scene to recover its class
    print(f'\n  Downloading .txt metadata for {len(drops)} dropped scenes (small files)...')
    DROP_TXT_DIR = '/dev/shm/dropped_scene_txts'
    os.makedirs(DROP_TXT_DIR, exist_ok=True)
    DOWNLOAD_SCRIPT = '/content/scannet_tools/download-scannet.py'
    if not os.path.isfile(DOWNLOAD_SCRIPT):
        os.makedirs('/content/scannet_tools', exist_ok=True)
        !wget -q -O {DOWNLOAD_SCRIPT} 'http://kaldir.vc.cit.tum.de/scannet/download-scannet.py'

    def _dl_txt(sid):
        out = os.path.join(DROP_TXT_DIR, 'scans', sid, f'{sid}.txt')
        if os.path.isfile(out):
            return sid, out, None
        try:
            r = subprocess.run(
                ['python3', DOWNLOAD_SCRIPT, '-o', DROP_TXT_DIR, '--id', sid, '--type', '.txt'],
                capture_output=True, text=True, timeout=30, input='y\ny\n'
            )
            if r.returncode == 0 and os.path.isfile(out):
                return sid, out, None
            return sid, None, f'rc={r.returncode}'
        except Exception as e:
            return sid, None, str(e)[:100]

    sid_to_txt = {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(_dl_txt, sid) for sid in drops.keys()]
        for f in as_completed(futs):
            sid, path, err = f.result()
            if path:
                sid_to_txt[sid] = path
    print(f'  Downloaded {len(sid_to_txt)}/{len(drops)} .txt files.')

    # Parse sceneType, normalize via the same alias map (try import from preprocess)
    def _parse_scene_type(txt_path):
        try:
            with open(txt_path) as f:
                for line in f:
                    if line.startswith('sceneType'):
                        return line.split('=', 1)[1].strip()
        except OSError:
            pass
        return ''

    # Build per-class drop count
    cls_drop_counter = Counter()
    raw_type_counter = Counter()
    for sid, txt in sid_to_txt.items():
        raw_type = _parse_scene_type(txt)
        raw_type_counter[raw_type] += 1
        # Best-effort canonicalization: lowercase, replace spaces with underscores
        canon = raw_type.lower().strip().replace(' ', '_').replace('/', '_')
        cls_drop_counter[canon] += 1

    print('\n  Dropped scenes by raw sceneType (top 20):')
    for raw, n in raw_type_counter.most_common(20):
        print(f'    {raw!r:30s}: {n}')

    # Compare to per-class scene counts on disk to estimate impact
    on_disk_scenes = {}
    for split in ['train', 'val']:
        sd = os.path.join(LOCAL_PATH, split)
        if not os.path.isdir(sd): continue
        for cls in os.listdir(sd):
            cd = os.path.join(sd, cls)
            if not os.path.isdir(cd): continue
            scenes = set()
            for fn in os.listdir(cd):
                m = re.match(r'(scene\d+_\d+)_f', fn)
                if m: scenes.add(m.group(1))
            on_disk_scenes[cls] = on_disk_scenes.get(cls, 0) + len(scenes)

    # Per-class drop count via the EXACT same canonicalizer the preprocess uses.
    # Try importing; fall back to a strict case-folded exact-match.
    # Strict case-folded exact match against on-disk class names
    print('\n  Per-class drop impact (strict exact-match canonicalization):')
    print(f'  {"class":<30}{"on disk":>10}{"dropped":>10}{"%":>8}')
    print('  ' + '-' * 60)
    impacts = []
    # First: build raw->canonical map by exact case-folded match against
    # on-disk class names. Anything that doesn't match is reported separately.
    on_disk_lower = {c.lower(): c for c in on_disk_scenes.keys()}
    raw_to_canon = {}
    unmatched_raw = Counter()
    for raw, n in raw_type_counter.items():
        key = raw.lower().strip().replace(' ', '_').replace('/', '_')
        if key in on_disk_lower:
            raw_to_canon[raw] = on_disk_lower[key]
        else:
            unmatched_raw[raw] = n
    # Tally drops per canonical class
    canon_drops = Counter()
    for raw, n in raw_type_counter.items():
        if raw in raw_to_canon:
            canon_drops[raw_to_canon[raw]] += n
    for cls in sorted(on_disk_scenes.keys()):
        kept = on_disk_scenes[cls]
        dropped = canon_drops.get(cls, 0)
        pct = 100 * dropped / max(kept + dropped, 1)
        impacts.append((cls, kept, dropped, pct))
        flag = '  <-- HIGH' if pct > 15 else ''
        print(f'  {cls:<30}{kept:>10}{dropped:>10}{pct:>7.1f}%{flag}')
    if unmatched_raw:
        print(f'\n  Unmatched raw sceneTypes (not assigned to any canonical class):')
        for raw, n in unmatched_raw.most_common(10):
            print(f'    {raw!r:30s}: {n}')
    max_impact = max((p for _,_,_,p in impacts), default=0)
    if max_impact > 20:
        verdict_6 = 'WARN'
        print(f'\n  WARN: at least one class lost >20% of scenes to the drop list.')
    else:
        verdict_6 = 'PASS'
    DIAG_RESULTS['drop_list'] = {
        'verdict': verdict_6,
        'total_dropped': len(drops),
        'max_class_loss_pct': max_impact,
    }
globals()['DIAG_RESULTS'] = DIAG_RESULTS

## 7. Extraction telemetry (Hypothesis D)
Read `processing_log.jsonl` to compute pose_dropped / blur_dropped / decode_failed fractions. Pre-HHA there was no pose check, so HHA mode silently has fewer training frames per scene.

In [ ]:
log_path = os.path.join(LOCAL_PATH, 'processing_log.jsonl')
if not os.path.isfile(log_path):
    print(f'No log at {log_path}.')
    print('processing_log.jsonl is written at the END of cell 26; if you restored from the')
    print('canonical tarball, the log isn\'t included. Skip this section.')
else:
    rows = []
    with open(log_path) as f:
        for line in f:
            rows.append(json.loads(line))
    print(f'Scenes in log: {len(rows)}')
    n_ok      = sum(1 for r in rows if r.get('status') == 'ok')
    # Detect both via status string AND fallback (pass2_attempted but 0 frames written).
    n_p2_fail = sum(
        1 for r in rows
        if r.get('status') == 'ok_pass2_failed'
        or (r.get('pass2_attempted') and r.get('pass2_frames', 0) == 0)
    )
    n_skip    = sum(1 for r in rows if r.get('status') == 'skipped')
    n_err     = sum(1 for r in rows if r.get('status') == 'error')
    print(f'  ok={n_ok}  ok_pass2_failed={n_p2_fail}  skipped={n_skip}  error={n_err}')
    total_frames = sum(r.get('num_frames', 0) for r in rows)
    total_blur_drop  = sum(r.get('blur_dropped', 0)  for r in rows)
    total_pose_drop  = sum(r.get('pose_dropped', 0)  for r in rows)
    total_decode_fail= sum(r.get('decode_failed', 0) for r in rows)
    total_attempted  = total_frames + total_blur_drop + total_pose_drop + total_decode_fail
    print(f'\n  Frames extracted:    {total_frames:,}')
    if total_attempted > 0:
        print(f'  Blur dropped:        {total_blur_drop:,}  ({100*total_blur_drop/total_attempted:.2f}%)')
        print(f'  Pose dropped:        {total_pose_drop:,}  ({100*total_pose_drop/total_attempted:.2f}%)')
        print(f'  Decode failed:       {total_decode_fail:,}  ({100*total_decode_fail/total_attempted:.2f}%)')
    if total_pose_drop / max(total_attempted, 1) > 0.05:
        print('\n  WARN: >5% of frames lost to pose-validity (HHA-only filter).')
        print('  Pre-HHA had no pose check, so HHA dataset is materially smaller.')
    # Per-class extraction loss
    per_cls = Counter()
    per_cls_drops = Counter()
    for r in rows:
        c = r.get('class_name')
        if not c: continue
        per_cls[c]      += r.get('num_frames', 0)
        per_cls_drops[c]+= r.get('pose_dropped', 0) + r.get('blur_dropped', 0)
    print('\n  Per-class extraction loss (drops / extracted):')
    for c in sorted(per_cls.keys()):
        kept = per_cls[c]; lost = per_cls_drops[c]; total = kept + lost
        if total > 0:
            pct = 100 * lost / total
            flag = '  <-- HIGH' if pct > 10 else ''
            print(f'    {c:30s} {kept:>6,} kept  {lost:>5,} lost  ({pct:5.2f}%){flag}')
    # Verdict dump
    DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
    pose_pct = 100 * total_pose_drop / max(total_attempted, 1)
    err_pct = 100 * n_err / max(len(rows), 1)
    DIAG_RESULTS['extraction_telemetry'] = {
        'verdict': 'WARN' if (pose_pct > 5 or err_pct > 5) else 'PASS',
        'pose_drop_pct': pose_pct,
        'err_pct': err_pct,
        'pass2_failures': n_p2_fail,
        'errors': n_err,
    }
    globals()['DIAG_RESULTS'] = DIAG_RESULTS

## 8. Combined NaN at train-time (Hypothesis E)
On-disk NaN rate is one thing; train-time NaN rate is higher because `apply_hha_aug` introduces NEW NaN regions via mask-aware blur and hole dropout. This is what the model actually trains on.

In [ ]:
# Run apply_hha_aug at typical training magnitudes; measure NaN before per-channel-mean replacement.
cfg = HHAAugConfig()  # explicit default; matches the dataset's training-time config
_rng8 = random.Random(42)
files = _rng8.sample(hha_files, 200)
rng_np = np.random.default_rng(42)
n_finite_pre = np.zeros(3, dtype=np.int64)
n_finite_post = np.zeros(3, dtype=np.int64)
n_total = np.zeros(3, dtype=np.int64)
n_components_pre = []
n_components_post = []
for p in files:
    h = torch.load(p, weights_only=True).float()
    h_aug = apply_hha_aug(h, cfg, rng=rng_np)
    # Count connected NaN regions (any-channel-NaN mask)
    mask_pre  = np.isnan(h.numpy()).any(axis=0)
    mask_post = np.isnan(h_aug.numpy()).any(axis=0)
    _, n_pre  = cc_label(mask_pre)
    _, n_post = cc_label(mask_post)
    n_components_pre.append(n_pre)
    n_components_post.append(n_post)
    for c in range(3):
        n_finite_pre[c]  += int(torch.isfinite(h[c]).sum())
        n_finite_post[c] += int(torch.isfinite(h_aug[c]).sum())
        n_total[c]       += int(h[c].numel())

print('On-disk vs post-augmentation NaN rate:')
for c, lab in enumerate(['disparity', 'height   ', 'angle    ']):
    pre  = 1.0 - n_finite_pre[c]  / n_total[c]
    post = 1.0 - n_finite_post[c] / n_total[c]
    delta = post - pre
    print(f'  ch{c} {lab}: on-disk={pre*100:5.2f}%  post-aug={post*100:5.2f}%  +{delta*100:.2f}pt')
max_post = max(1.0 - n_finite_post[c]/n_total[c] for c in range(3))
print(f'\nWorst post-aug NaN rate: {max_post*100:.2f}%')
med_cc_pre = float(np.median(n_components_pre)); med_cc_post = float(np.median(n_components_post))
print(f'\nNaN connected-component count (median per frame):')
print(f'  on-disk:  {med_cc_pre:.0f} components')
print(f'  post-aug: {med_cc_post:.0f} components (lower = bigger blob holes = worse for model)')

# Sweep hole_p to give an actionable knob: how much does post-aug NaN drop
# if we ease augmentation aggressiveness?
print('\nhole_p sweep (post-aug NaN rate by hole-dropout aggressiveness):')
_sweep_files = files[:60]  # smaller sample for speed
_sweep_rng = np.random.default_rng(42)
for _hp in (0.0, 0.15, 0.30, 0.50):
    cfg_s = HHAAugConfig(hole_p=_hp)
    n_fin_s = np.zeros(3, dtype=np.int64); n_tot_s = np.zeros(3, dtype=np.int64)
    for p in _sweep_files:
        h = torch.load(p, weights_only=True).float()
        h_aug = apply_hha_aug(h, cfg_s, rng=_sweep_rng)
        for c in range(3):
            n_fin_s[c] += int(torch.isfinite(h_aug[c]).sum())
            n_tot_s[c] += int(h_aug[c].numel())
    nan_rates = [(1.0 - n_fin_s[c]/n_tot_s[c])*100 for c in range(3)]
    print(f'  hole_p={_hp:.2f} -> ch0/1/2 NaN: {nan_rates[0]:5.2f}% / {nan_rates[1]:5.2f}% / {nan_rates[2]:5.2f}%')
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
# Verdict: blob-shaped (low cc count) + high NaN rate is worse than scattered.
if max_post > 0.20 and med_cc_post < 10:
    _v8 = 'FAIL'
elif max_post > 0.25:
    _v8 = 'WARN'
elif max_post > 0.15:
    _v8 = 'INFO'
else:
    _v8 = 'PASS'
DIAG_RESULTS['train_time_nan'] = {
    'verdict': _v8,
    'max_post_aug_nan_rate': max_post,
    'median_nan_components_post_aug': med_cc_post,
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS
if max_post > 0.25:
    print('  WARN: >25% of HHA pixels become per-channel-mean (then ~0 after normalize)')
    print('  during training. Depth-stream signal is heavily diluted. Consider lowering')
    print('  hole_p / hole_size_max in HHAAugConfig, or lowering BASE_BLUR_P.')

## 9. Distribution vs SUN HHA

In [ ]:
sun_paths = [
    '/content/sunrgbd_19_hha/norm_stats.json',
    '/content/Multi-Stream-Neural-Networks/data/sunrgbd_19_hha/norm_stats.json',
    '/content/drive/MyDrive/datasets/sunrgbd_19_hha/norm_stats.json',
]
# If only the tarball is on Drive, extract metadata-only quickly
if not any(os.path.isfile(p) for p in sun_paths):
    sun_tar = '/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz'
    if os.path.isfile(sun_tar):
        print(f'Extracting SUN norm_stats from {sun_tar}...')
        !mkdir -p /content/sunrgbd_19_hha && cd /content/sunrgbd_19_hha && tar -xzf {sun_tar} --wildcards '*/norm_stats.json' --strip-components=1 2>/dev/null
sun_path = next((p for p in sun_paths if os.path.isfile(p)), None)
if NS is None:
    print('Skipping vs-SUN comparison: ScanNet HHA NS missing (see section 2).')
    DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
    DIAG_RESULTS['vs_sun'] = {'verdict': 'INCONCLUSIVE', 'reason': 'ScanNet NS missing'}
    globals()['DIAG_RESULTS'] = DIAG_RESULTS
elif sun_path is None:
    print('SUN norm_stats.json not found locally or on Drive at standard paths.')
else:
    with open(sun_path) as f:
        SUN = json.load(f)
    print(f'Comparing ScanNet vs SUN HHA stats:')
    print(f"{'Channel':<22}{'ScanNet':>14}{'SUN':>14}{'|d|/SUN':>10}")
    print('-' * 62)
    max_rel = 0.0
    for c, lab in enumerate(['disparity (1/m)', 'height (m)    ', 'angle (deg)   ']):
        sc_m = NS['hha_mean'][c]; su_m = SUN['hha_mean'][c]
        sc_s = NS['hha_std'][c];  su_s = SUN['hha_std'][c]
        rel_m = abs(sc_m - su_m) / max(abs(su_m), 1e-6)
        rel_s = abs(sc_s - su_s) / max(abs(su_s), 1e-6)
        max_rel = max(max_rel, rel_m, rel_s)
        print(f'  {lab} mean    {sc_m:14.4f}{su_m:14.4f}{rel_m*100:9.1f}%')
        print(f'  {lab} std     {sc_s:14.4f}{su_s:14.4f}{rel_s*100:9.1f}%')
    if 'hha_ranges' in NS and 'hha_ranges' in SUN:
        print(f"\n{'Channel':<22}{'ScanNet nan':>14}{'SUN nan':>14}")
        print('-' * 50)
        for c, lab in enumerate(['disparity', 'height   ', 'angle    ']):
            sc_n = NS['hha_ranges'][c]['nan_rate']; su_n = SUN['hha_ranges'][c]['nan_rate']
            print(f'  ch{c} {lab}     {sc_n*100:13.2f}%{su_n*100:13.2f}%')
    print(f'\nMax relative divergence: {max_rel*100:.1f}%')
    DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
    DIAG_RESULTS['vs_sun'] = {
        'verdict': 'WARN' if max_rel > 0.30 else 'PASS',
        'max_relative_divergence': max_rel,
    }
    globals()['DIAG_RESULTS'] = DIAG_RESULTS
    if max_rel > 0.30:
        print('  HHA distributions diverge >30% from SUN. Pretrained features will be')
        print('  off-distribution at fine-tune time.')

## 9b. Raw-depth ScanNet baseline comparison

If you still have the pre-HHA `scannet_pretrain_256.tar.gz` on Drive, compare its `norm_stats.json` to the HHA dataset's. Identical RGB stats = same scene population (sanity); divergent RGB stats = HHA preprocessing dropped scenes the raw-depth pipeline kept (scene-loss explains some of the regression by itself).

In [ ]:
RAW_DEPTH_NS_PATHS = [
    '/content/drive/MyDrive/datasets/scannet_pretrain_256/norm_stats.json',
    '/content/scannet_pretrain_256/norm_stats.json',
]
raw_ns_path = next((p for p in RAW_DEPTH_NS_PATHS if os.path.isfile(p)), None)
raw_tar_path = '/content/drive/MyDrive/datasets/scannet_pretrain_256.tar.gz'
verdict_9b = 'INCONCLUSIVE'
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
if raw_ns_path is None and os.path.isfile(raw_tar_path):
    print(f'Extracting raw-depth norm_stats from {raw_tar_path} (metadata only)...')
    !mkdir -p /content/scannet_pretrain_256 && cd /content/scannet_pretrain_256 && tar -xzf {raw_tar_path} --wildcards '*/norm_stats.json' --strip-components=1 2>/dev/null || true
    raw_ns_path = next((p for p in RAW_DEPTH_NS_PATHS if os.path.isfile(p)), None)
if raw_ns_path is None:
    print(f'Raw-depth dataset not found on Drive at {raw_tar_path}.')
    print('Skip if you did not retain the pre-HHA tarball.')
    DIAG_RESULTS['raw_depth_baseline'] = {'verdict': verdict_9b, 'reason': 'raw tarball not found'}
elif NS is None:
    print('Skipping: ScanNet HHA norm_stats.json missing (see section 2).')
    DIAG_RESULTS['raw_depth_baseline'] = {'verdict': verdict_9b, 'reason': 'HHA NS missing'}
else:
    with open(raw_ns_path) as f:
        RAW = json.load(f)
    print(f'Raw-depth norm_stats: {raw_ns_path}')
    print(f"\n{'Channel':<22}{'HHA-run':>14}{'Raw-run':>14}{'rel diff':>10}")
    print('-' * 60)
    rgb_max_rel = 0.0
    for c, lab in enumerate(['RGB R mean', 'RGB G mean', 'RGB B mean']):
        hha_v = NS['rgb_mean'][c]; raw_v = RAW['rgb_mean'][c]
        rel = abs(hha_v - raw_v) / max(abs(raw_v), 1e-6)
        rgb_max_rel = max(rgb_max_rel, rel)
        print(f'{lab:<22}{hha_v:>14.4f}{raw_v:>14.4f}{rel*100:>9.2f}%')
    # Also compare frame counts (the most direct measure of scene loss)
    raw_info_path = os.path.dirname(raw_ns_path)
    raw_train_dir = os.path.join(raw_info_path, 'train')
    raw_n_train = None
    if os.path.isdir(raw_train_dir):
        raw_n_train = sum(1 for _ in Path(raw_train_dir).rglob('*_rgb.pt'))
    print(f'\nFrame-count comparison:')
    print(f'  HHA-run train frames:      {n_train:,}')
    if raw_n_train is not None:
        print(f'  Raw-depth-run train frames: {raw_n_train:,}')
        delta = (n_train - raw_n_train) / max(raw_n_train, 1)
        print(f'  Delta:                     {delta*100:+.1f}%')
        if delta < -0.05:
            print(f'  WARN: HHA pipeline produced {-delta*100:.1f}% fewer frames than raw-depth.')
    else:
        print(f'  Raw-depth train dir not present (only norm_stats was extracted).')
    if rgb_max_rel < 0.0001:
        verdict_9b = 'PASS'
        print(f'\n  PASS: RGB stats are essentially identical (rel diff {rgb_max_rel*100:.4f}%).')
        print(f'  Drop list and pose-rejection had effectively zero scene-population impact.')
    elif rgb_max_rel < 0.01:
        verdict_9b = 'PASS'
        print(f'\n  PASS: RGB stats match within {rgb_max_rel*100:.2f}%. Same scene population.')
    else:
        verdict_9b = 'WARN'
        print(f'\n  WARN: RGB stats differ by up to {rgb_max_rel*100:.2f}%.')
        print(f'  HHA preprocessing dropped or added scenes -- not just a depth representation change.')
        print(f'  Scene-population shift contributes to the val regression independently of HHA quality.')
    DIAG_RESULTS['raw_depth_baseline'] = {
        'verdict': verdict_9b,
        'rgb_max_rel_diff': rgb_max_rel,
    }
globals()['DIAG_RESULTS'] = DIAG_RESULTS

## 10. Pass 2 frame correlation
Pass 2 selects frames `MIN_GAP_FRAMES=11` apart from Pass 1 — at 30 Hz that's 0.37s. For rare-class scenes, consecutive Pass 1 + Pass 2 frames may be visually near-duplicate. Measure RGB cosine similarity between adjacent extracted frames in rare-class scenes.

In [ ]:
# Load class names; identify rare classes (<50 scenes); compute median RGB cosine
# similarity between consecutive frames in same scene.
with open(os.path.join(LOCAL_PATH, 'class_names.txt')) as f:
    class_names = [l.strip() for l in f if l.strip()]

# Per-class scene counts on disk
per_cls_scenes = {}
for split in ['train', 'val']:
    sd = os.path.join(LOCAL_PATH, split)
    if not os.path.isdir(sd): continue
    for cls in os.listdir(sd):
        cd = os.path.join(sd, cls)
        if not os.path.isdir(cd): continue
        scenes = set()
        for f in os.listdir(cd):
            m = re.match(r'(scene\d+_\d+)_f', f)
            if m: scenes.add(m.group(1))
        per_cls_scenes[cls] = per_cls_scenes.get(cls, 0) + len(scenes)

PASS2_RARE_THRESHOLD = 50  # matches preprocess cell 6
rare_classes = [c for c, n in per_cls_scenes.items() if n < PASS2_RARE_THRESHOLD]
common_classes = [c for c, n in per_cls_scenes.items() if n >= PASS2_RARE_THRESHOLD]
print(f'Rare classes (<50 scenes): {sorted(rare_classes)}')
print(f'Common classes (>=50): {sorted(common_classes)[:5]}... (showing 5)')

def adjacent_cosine(class_dir, max_pairs=200):
    """For each scene in class_dir, compute cosine similarity between
    consecutive-by-frame-idx RGB tensors. Returns list of similarities."""
    sims = []
    by_scene = {}
    for f in os.listdir(class_dir):
        if not f.endswith('_rgb.pt'): continue
        m = re.match(r'(scene\d+_\d+)_f(\d+)_rgb\.pt', f)
        if not m: continue
        sid, idx = m.group(1), int(m.group(2))
        by_scene.setdefault(sid, []).append((idx, os.path.join(class_dir, f)))
    PER_SCENE_CAP = 10  # balance sample across many scenes; not biased by 1-2 dominators
    for sid, lst in sorted(by_scene.items())[:30]:
        lst.sort(key=lambda x: x[0])
        scene_sims = 0
        for (i1, p1), (i2, p2) in zip(lst[:-1], lst[1:]):
            if scene_sims >= PER_SCENE_CAP: break
            r1 = torch.load(p1, weights_only=True).float().flatten()
            r2 = torch.load(p2, weights_only=True).float().flatten()
            # Mean-centered cosine = structural similarity, not DC brightness.
            r1c = r1 - r1.mean(); r2c = r2 - r2.mean()
            cos = float((r1c @ r2c) / (r1c.norm() * r2c.norm() + 1e-8))
            sims.append(cos); scene_sims += 1
    return sims

cls_medians = {}
cls_sims = {}  # for per-class histogram plot
for tag, cls_set in [('rare', rare_classes[:3]), ('common', common_classes[:3])]:
    print(f'\n{tag}-class adjacent-frame RGB cosine similarity:')
    for cls in cls_set:
        all_sims = []
        for split in ['train', 'val']:
            cd = os.path.join(LOCAL_PATH, split, cls)
            if not os.path.isdir(cd): continue
            all_sims.extend(adjacent_cosine(cd))
        if all_sims:
            arr = np.array(all_sims)
            cls_medians[(tag, cls)] = float(np.median(arr))
            cls_sims[(tag, cls)] = arr
            print(f'  {cls:25s}: n={len(arr):4d}  median={np.median(arr):.4f}  '
                  f'p99={np.percentile(arr,99):.4f}  >0.99={(arr>0.99).mean()*100:.1f}%')
print('\nInterpretation:')
print('  median cos > 0.95 = adjacent frames very similar (low diversity)')
print('  median cos > 0.99 = effectively duplicate frames; Pass 2 packed too tight')
rare_medians = [m for (t, _), m in cls_medians.items() if t == 'rare']
common_medians = [m for (t, _), m in cls_medians.items() if t == 'common']
worst_rare = max(rare_medians) if rare_medians else 0.0
med_rare = float(np.median(rare_medians)) if rare_medians else 0.0
print(f'\n  Rare-class medians: worst={worst_rare:.4f}  median-of-medians={med_rare:.4f}')
if worst_rare > 0.99:
    verdict_10 = 'FAIL'
    print(f'\n  FAIL: rare-class median similarity {worst_rare:.4f} > 0.99 -- effective duplicates.')
elif worst_rare > 0.97:
    verdict_10 = 'WARN'
    print(f'\n  WARN: rare-class median similarity {worst_rare:.4f} > 0.97 -- frames are very similar.')
elif rare_medians:
    verdict_10 = 'PASS'
    print(f'\n  PASS: max rare-class median {worst_rare:.4f}; adequate diversity.')
else:
    verdict_10 = 'INCONCLUSIVE'
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
# Per-class histogram of cosine similarities; vertical line at 0.99 = duplicate threshold
if cls_sims:
    fig, ax = plt.subplots(figsize=(9, 4))
    for (tag, cls), arr in cls_sims.items():
        ax.hist(arr, bins=30, alpha=0.4, label=f'{tag}/{cls}', density=True)
    ax.axvline(0.99, color='red', linestyle='--', label='0.99 (dup threshold)')
    ax.axvline(0.97, color='orange', linestyle=':', label='0.97 (warn threshold)')
    ax.set_xlabel('mean-centered cosine similarity'); ax.set_ylabel('density')
    ax.legend(loc='upper left'); ax.set_title('Adjacent-frame similarity distributions')
    plt.tight_layout(); plt.show(); plt.close(fig)
DIAG_RESULTS['pass2_correlation'] = {
    'verdict': verdict_10,
    'worst_rare_median': worst_rare,
    'median_rare_median': med_rare,
    'rare_medians': rare_medians,
    'common_medians': common_medians,
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS

## 11. Float16 round-trip sanity

In [ ]:
# Recompute mean/std from float16 on-disk tensors; compare to norm_stats.json.
_rng11 = random.Random(42)
files = _rng11.sample(hha_files, 200)
sums = np.zeros(3, dtype=np.float64)
sumsqs = np.zeros(3, dtype=np.float64)
ns = np.zeros(3, dtype=np.int64)
dtypes = Counter()
for p in files:
    h = torch.load(p, weights_only=True)
    dtypes[str(h.dtype)] += 1
    h32 = h.float().numpy()
    for c in range(3):
        v = h32[c][np.isfinite(h32[c])].astype(np.float64)
        sums[c]   += v.sum(); sumsqs[c] += (v ** 2).sum(); ns[c] += v.size
mean_emp = sums / ns
var_emp  = (sumsqs - ns * mean_emp ** 2) / np.maximum(ns - 1, 1)
std_emp  = np.sqrt(np.clip(var_emp, 0, None))
print(f'Sampled tensor dtypes: {dict(dtypes)}')
print(f"\n{'Channel':<22}{'on-disk mean':>14}{'norm_stats':>14}{'rel %':>10}")
print('-' * 62)
for c, lab in enumerate(['disparity', 'height   ', 'angle    ']):
    pub_m = NS['hha_mean'][c]
    rel = abs(mean_emp[c] - pub_m) / max(abs(pub_m), 1e-6) * 100
    print(f'  ch{c} {lab}    {mean_emp[c]:14.6f}{pub_m:14.6f}{rel:9.4f}%')
for c, lab in enumerate(['disparity', 'height   ', 'angle    ']):
    pub_s = NS['hha_std'][c]
    rel = abs(std_emp[c] - pub_s) / max(abs(pub_s), 1e-6) * 100
    print(f'  ch{c} {lab}std {std_emp[c]:14.6f}{pub_s:14.6f}{rel:9.4f}%')
max_rel_mean = max(
    abs(mean_emp[c] - NS['hha_mean'][c]) / max(abs(NS['hha_mean'][c]), 1e-6) for c in range(3)
)
max_rel_std = max(
    abs(std_emp[c]  - NS['hha_std'][c])  / max(abs(NS['hha_std'][c]),  1e-6) for c in range(3)
)
max_rel = max(max_rel_mean, max_rel_std)
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
DIAG_RESULTS['float16_roundtrip'] = {
    'verdict': 'WARN' if max_rel > 0.01 else 'PASS',
    'max_rel_diff_mean': max_rel_mean,
    'max_rel_diff_std':  max_rel_std,
    'max_rel_diff':      max_rel,
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS
print()
if max_rel > 0.01:
    print(f'WARN: max relative difference is {max_rel*100:.2f}% (mean={max_rel_mean*100:.2f}%, std={max_rel_std*100:.2f}%).')
    print('  Possible causes: norm_stats sample population differs from current files')
    print('  (e.g., cells 37/39 deleted files after norm_stats ran), OR sample size of')
    print('  200 is too small for tight std convergence. Run with all files to confirm.')
else:
    print(f'PASS: max relative difference is {max_rel*100:.4f}% (tight match).')

## 12. Sample visualization (sanity gut-check)

In [ ]:
_rng12 = random.Random(42)
viz_files = _rng12.sample(hha_files, 6)
_v4 = DIAG_RESULTS.get('floor_angle', {}).get('verdict', '?')
_flip_warn = ' [SECTION 4 SAID INVERTED]' if _v4 == 'FAIL' else ''
fig, axes = plt.subplots(6, 4, figsize=(14, 18))
for row, hp in enumerate(viz_files):
    rp = hp.replace('_hha.pt', '_rgb.pt')
    cls = os.path.basename(os.path.dirname(hp))
    rgb = torch.load(rp, weights_only=True).permute(1, 2, 0).numpy()
    h = torch.load(hp, weights_only=True).float().numpy()
    nan_overlay = np.isnan(h).any(axis=0)
    axes[row, 0].imshow(rgb); axes[row, 0].set_title(f'{cls}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(np.nan_to_num(h[0], nan=0.0), cmap='viridis', vmin=0, vmax=3.0)
    axes[row, 1].imshow(nan_overlay, cmap='Reds', alpha=0.4*nan_overlay)
    axes[row, 1].set_title('disparity (red=NaN)'); axes[row, 1].axis('off')
    axes[row, 2].imshow(np.nan_to_num(h[1], nan=0.0), cmap='RdBu_r', vmin=-2, vmax=2)
    axes[row, 2].imshow(nan_overlay, cmap='Reds', alpha=0.4*nan_overlay)
    axes[row, 2].set_title('height (red=NaN)'); axes[row, 2].axis('off')
    norm = TwoSlopeNorm(vmin=0, vcenter=90, vmax=180)
    axes[row, 3].imshow(np.nan_to_num(h[2], nan=90.0), cmap='RdBu_r', norm=norm)
    axes[row, 3].imshow(nan_overlay, cmap='Reds', alpha=0.4*nan_overlay)
    axes[row, 3].set_title(f'angle [0=ceil 180=floor] (red=NaN){_flip_warn}'); axes[row, 3].axis('off')
plt.tight_layout(); plt.show(); plt.close(fig)

## 15. Hypothesis panel — floor / clamp / angle (SUN vs ScanNet asymmetries)

Three targeted checks for the SUN→ScanNet HHA-transfer regression. Each can independently confirm or rule out one of the audit hypotheses.

- **H1 — floor reference noise.** `compute_hha` uses `floor = percentile(Z_world, 5)` per frame. SUN frames almost always contain floor; ScanNet handheld captures often don't. We measure the per-frame fraction of pixels within 15 cm of floor (the *near-floor frac*). If ScanNet has many frames with near-floor frac under 5%, the height channel's zero-point is effectively random on those frames.
- **H2 — 8 m depth clamp.** `_MAX_DEPTH_M = 8.0` is a SUN constant. Disparity saturates at `1/8 = 0.125`. We measure the per-frame fraction of pixels saturated at the clamp, broken down by class. Large-space classes (`hallway`, `lobby`, `gym`, `bookstore_library`) should show much higher saturation if this is biting.
- **H3 — angle channel peaks.** Canonical HHA has bimodal/trimodal peaks at 0° (ceilings), 90° (walls), 180° (floors). Compare ScanNet vs SUN. Diffuse / shifted peaks point at a residual rotation-convention issue not caught by the Z-only Phase 0 test.

Run this before §13 so the verdict-summary cell picks up the results.


In [ ]:
# === §15. Hypothesis panel ============================================
# Tests H1 (floor visibility), H2 (8m clamp), H3 (angle bimodal peaks).
# Reuses hha_files, LOCAL_PATH, NS from earlier sections.
#
# Verdict semantics:
#   CONFIRMED = strong asymmetry vs SUN; this hypothesis explains a chunk of the drop.
#   PARTIAL   = real but small effect.
#   REJECTED  = no significant asymmetry; this isn't the culprit.
#   INCONCLUSIVE = SUN reference data missing or sample too small.
# ====================================================================
_rng15 = random.Random(42)
SAMPLE_N = 400

# ---- Load SUN HHA tensor once (used by H1 + H3) ----------------------
sun_paths = [
    '/content/sunrgbd_19_hha/train/hha_tensors.pt',
    '/content/Multi-Stream-Neural-Networks/data/sunrgbd_19_hha/train/hha_tensors.pt',
]
sun_path = next((p for p in sun_paths if os.path.isfile(p)), None)
if sun_path is None:
    sun_tar = '/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz'
    if os.path.isfile(sun_tar):
        print(f'Extracting SUN HHA tensors from {sun_tar} (one-off, ~3 GB)...')
        os.makedirs('/content/sunrgbd_19_hha', exist_ok=True)
        subprocess.run(
            ['tar', '-xzf', sun_tar, '-C', '/content/sunrgbd_19_hha',
             '--wildcards', '*/train/hha_tensors.pt', '--strip-components=1'],
            check=False, capture_output=True,
        )
        sun_path = next((p for p in sun_paths if os.path.isfile(p)), None)
sun_t = None
if sun_path is not None:
    sun_t = torch.load(sun_path, weights_only=True, map_location='cpu')
    print(f'Loaded SUN HHA tensor: {sun_t.shape} {sun_t.dtype} from {sun_path}')
else:
    print('SUN hha_tensors.pt not available; H1/H3 cross-dataset comparison will be INCONCLUSIVE.')

# ---- H1: per-frame floor visibility ----------------------------------
files_sn = _rng15.sample(hha_files, min(SAMPLE_N, len(hha_files)))
near_floor_sn = []
height_std_sn = []
for p in files_sn:
    h = torch.load(p, weights_only=True).float().numpy()
    height = h[1]
    finite = np.isfinite(height)
    if finite.sum() < 100:
        continue
    h_valid = height[finite]
    near_floor_sn.append(float(np.mean(np.abs(h_valid) < 0.15)))
    height_std_sn.append(float(h_valid.std()))
near_floor_sn = np.array(near_floor_sn)
height_std_sn = np.array(height_std_sn)

near_floor_su = None
height_std_su = None
if sun_t is not None:
    idx = _rng15.sample(range(sun_t.shape[0]), min(SAMPLE_N, sun_t.shape[0]))
    near_floor_su = []
    height_std_su = []
    for i in idx:
        h = sun_t[i].float().numpy()
        height = h[1]
        finite = np.isfinite(height)
        if finite.sum() < 100:
            continue
        h_valid = height[finite]
        near_floor_su.append(float(np.mean(np.abs(h_valid) < 0.15)))
        height_std_su.append(float(h_valid.std()))
    near_floor_su = np.array(near_floor_su)
    height_std_su = np.array(height_std_su)

print('=' * 70)
print('H1 — per-frame floor visibility (fraction of pixels with |height| < 0.15m)')
print('=' * 70)
def _summary_h1(arr, name):
    print(f'  {name:<10} n={len(arr):4d}  median={np.median(arr):.3f}  '
          f'p10={np.percentile(arr,10):.3f}  p90={np.percentile(arr,90):.3f}  '
          f'<5%={float((arr<0.05).mean())*100:5.1f}%  <2%={float((arr<0.02).mean())*100:5.1f}%')
_summary_h1(near_floor_sn, 'ScanNet:')
if near_floor_su is not None:
    _summary_h1(near_floor_su, 'SUN:')

print(f'  Per-frame height STD (m):')
print(f'    ScanNet median={np.median(height_std_sn):.3f}  IQR=[{np.percentile(height_std_sn,25):.3f}, {np.percentile(height_std_sn,75):.3f}]')
if height_std_su is not None:
    print(f'    SUN     median={np.median(height_std_su):.3f}  IQR=[{np.percentile(height_std_su,25):.3f}, {np.percentile(height_std_su,75):.3f}]')

verdict_h1 = 'INCONCLUSIVE'
if near_floor_su is not None:
    sn_no_floor = float((near_floor_sn < 0.05).mean())
    su_no_floor = float((near_floor_su < 0.05).mean())
    if sn_no_floor > su_no_floor * 2.0 + 0.05:
        verdict_h1 = 'CONFIRMED'
    elif sn_no_floor > su_no_floor + 0.05:
        verdict_h1 = 'PARTIAL'
    else:
        verdict_h1 = 'REJECTED'
    print(f'\n  H1 verdict: {verdict_h1}  (ScanNet floor-blind {sn_no_floor*100:.1f}%  vs  SUN {su_no_floor*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(near_floor_sn, bins=40, alpha=0.5, label='ScanNet', density=True, color='C0')
if near_floor_su is not None:
    ax.hist(near_floor_su, bins=40, alpha=0.5, label='SUN', density=True, color='C1')
ax.axvline(0.05, color='red', linestyle='--', label='5% threshold (floor invisible)')
ax.set_xlabel('Per-frame near-floor fraction (|height| < 0.15 m)')
ax.set_ylabel('Density')
ax.set_title('H1: per-frame floor visibility')
ax.legend(); plt.tight_layout(); plt.show(); plt.close(fig)

# ---- H2: 8m depth clamp saturation per class -------------------------
# Disparity = 1/depth, clamped at 1/8 = 0.125. A clamped pixel reads ~0.125.
CLAMP_EPS = 1e-3
per_class_clamp = {}
train_root = os.path.join(LOCAL_PATH, 'train')
for cls in sorted(os.listdir(train_root)) if os.path.isdir(train_root) else []:
    cls_dir = os.path.join(train_root, cls)
    if not os.path.isdir(cls_dir):
        continue
    cls_files = [f for f in os.listdir(cls_dir) if f.endswith('_hha.pt')]
    if not cls_files:
        continue
    sample_cls = _rng15.sample(cls_files, min(30, len(cls_files)))
    fracs = []
    for fname in sample_cls:
        h = torch.load(os.path.join(cls_dir, fname), weights_only=True).float().numpy()
        disp = h[0]
        finite = np.isfinite(disp)
        if finite.sum() < 100:
            continue
        clamped = np.abs(disp[finite] - 0.125) < CLAMP_EPS
        fracs.append(float(clamped.mean()))
    if fracs:
        per_class_clamp[cls] = (float(np.median(fracs)), float(np.mean(fracs)), len(fracs))

AUDIT_KEYS = ('hallway', 'lobby', 'gym', 'bookstore', 'library')
print('\n' + '=' * 70)
print('H2 — 8m clamp saturation per class (per-frame fraction of clamped pixels)')
print('=' * 70)
print(f'  {"class":<26}{"median":>10}{"mean":>10}{"n":>6}')
for cls, (med, mn, n) in sorted(per_class_clamp.items(), key=lambda kv: -kv[1][0]):
    flag = '  <-- audit-flagged (large room)' if any(k in cls for k in AUDIT_KEYS) else ''
    print(f'  {cls:<26}{med*100:9.2f}%{mn*100:9.2f}%{n:6d}{flag}')

audit_classes = [c for c in per_class_clamp if any(k in c for k in AUDIT_KEYS)]
other_classes = [c for c in per_class_clamp if c not in audit_classes]
audit_med = float(np.mean([per_class_clamp[c][0] for c in audit_classes])) if audit_classes else 0.0
other_med = float(np.mean([per_class_clamp[c][0] for c in other_classes])) if other_classes else 0.0
print(f'  Audit-flagged class mean-of-medians: {audit_med*100:.2f}%')
print(f'  Other classes      mean-of-medians: {other_med*100:.2f}%')
verdict_h2 = 'INCONCLUSIVE'
if audit_classes and other_classes:
    ratio = audit_med / max(other_med, 1e-6)
    if audit_med > other_med * 2.0 and audit_med > 0.02:
        verdict_h2 = 'CONFIRMED'
    elif audit_med > other_med + 0.01:
        verdict_h2 = 'PARTIAL'
    else:
        verdict_h2 = 'REJECTED'
    print(f'\n  H2 verdict: {verdict_h2}  (audit/other clamp ratio = {ratio:.2f}x)')

# ---- H3: angle channel bimodal peaks ---------------------------------
def _collect_angles(load_iter, n_frames, sub_per_frame=50_000):
    angs = []
    rng = np.random.default_rng(0)
    for k, h in enumerate(load_iter):
        if k >= n_frames:
            break
        a = h[2]; a = a[np.isfinite(a)]
        if a.size > sub_per_frame:
            a = rng.choice(a, sub_per_frame, replace=False)
        angs.append(a)
    return np.concatenate(angs) if angs else np.array([])

def _peak_frac(a, center, half=8):
    if a.size == 0:
        return float('nan')
    return float(np.mean((a >= center - half) & (a <= center + half)))

sn_iter_files = _rng15.sample(hha_files, min(300, len(hha_files)))
def _iter_sn():
    for p in sn_iter_files:
        yield torch.load(p, weights_only=True).float().numpy()
ang_sn = _collect_angles(_iter_sn(), 300)

ang_su = None
if sun_t is not None:
    su_idx = _rng15.sample(range(sun_t.shape[0]), min(300, sun_t.shape[0]))
    def _iter_su():
        for i in su_idx:
            yield sun_t[i].float().numpy()
    ang_su = _collect_angles(_iter_su(), 300)

print('\n' + '=' * 70)
print('H3 — angle channel bimodal peaks (fraction of pixels in ±8° of each peak)')
print('=' * 70)
print(f'  {"dataset":<10}{"peak@0 (ceil)":>16}{"peak@90 (wall)":>17}{"peak@180 (floor)":>19}')
print(f'  {"ScanNet:":<10}{_peak_frac(ang_sn, 0)*100:14.2f}%  {_peak_frac(ang_sn, 90)*100:14.2f}%  {_peak_frac(ang_sn, 180)*100:16.2f}%')
if ang_su is not None:
    print(f'  {"SUN:":<10}{_peak_frac(ang_su, 0)*100:14.2f}%  {_peak_frac(ang_su, 90)*100:14.2f}%  {_peak_frac(ang_su, 180)*100:16.2f}%')

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(ang_sn, bins=90, range=(0, 180), alpha=0.5, label='ScanNet', density=True, color='C0')
if ang_su is not None:
    ax.hist(ang_su, bins=90, range=(0, 180), alpha=0.5, label='SUN', density=True, color='C1')
for x, lab in [(0, 'ceiling'), (90, 'wall'), (180, 'floor')]:
    ax.axvline(x, color='gray', linestyle=':', alpha=0.6)
    ax.text(x, ax.get_ylim()[1] * 0.95, lab, rotation=90, va='top', ha='right', fontsize=8, color='gray')
ax.set_xlabel('Angle with gravity (deg)')
ax.set_ylabel('Density')
ax.set_title('H3: angle-channel pixel histogram')
ax.legend(); plt.tight_layout(); plt.show(); plt.close(fig)

verdict_h3 = 'INCONCLUSIVE'
if ang_su is not None:
    sn_floor = _peak_frac(ang_sn, 180)
    su_floor = _peak_frac(ang_su, 180)
    sn_ceil  = _peak_frac(ang_sn, 0)
    su_ceil  = _peak_frac(ang_su, 0)
    # A composition bug typically flattens/shifts the floor peak.
    if sn_floor < 0.5 * su_floor and sn_floor < 0.05:
        verdict_h3 = 'PARTIAL'
    elif abs(sn_floor - su_floor) < max(0.02, 0.25 * su_floor):
        verdict_h3 = 'REJECTED'
    else:
        verdict_h3 = 'PARTIAL'
    print(f'\n  H3 verdict: {verdict_h3}  (floor peak ScanNet {sn_floor*100:.1f}%  vs  SUN {su_floor*100:.1f}%; ceiling {sn_ceil*100:.1f}% vs {su_ceil*100:.1f}%)')

# ---- record + summarize ---------------------------------------------
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
DIAG_RESULTS['h1_floor_visibility'] = {
    'verdict': verdict_h1,
    'scannet_frac_floor_blind_below_5pct': float((near_floor_sn < 0.05).mean()),
    'sun_frac_floor_blind_below_5pct': float((near_floor_su < 0.05).mean()) if near_floor_su is not None else None,
    'scannet_height_std_median': float(np.median(height_std_sn)),
    'sun_height_std_median': float(np.median(height_std_su)) if height_std_su is not None else None,
}
DIAG_RESULTS['h2_depth_clamp'] = {
    'verdict': verdict_h2,
    'audit_class_clamp_median': audit_med,
    'other_class_clamp_median': other_med,
    'per_class_clamp_median': {c: per_class_clamp[c][0] for c in per_class_clamp},
}
DIAG_RESULTS['h3_angle_peaks'] = {
    'verdict': verdict_h3,
    'scannet_peak_180': _peak_frac(ang_sn, 180),
    'scannet_peak_90':  _peak_frac(ang_sn, 90),
    'scannet_peak_0':   _peak_frac(ang_sn, 0),
    'sun_peak_180':     _peak_frac(ang_su, 180) if ang_su is not None else None,
    'sun_peak_90':      _peak_frac(ang_su, 90)  if ang_su is not None else None,
    'sun_peak_0':       _peak_frac(ang_su, 0)   if ang_su is not None else None,
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS
print(f'\n§15 verdicts: H1={verdict_h1}  H2={verdict_h2}  H3={verdict_h3}')

# ---- Compact CLI-style summary (mirrors compare_hha_sun_scannet.py) --
# Self-contained block so it can be re-run / read on its own. Recomputes
# the few extras the per-hypothesis blocks above don't surface: 6-bin
# angle histograms, per-channel std/p1/p99 over the FULL sampled pool,
# and the NaN-fraction-per-frame mean.
print('\n' + '=' * 64)
print('§15 CLI-STYLE SUMMARY (mirrors compare_hha_sun_scannet.py output)')
print('=' * 64)

def _ch_stats(load_iter, n_frames):
    sums = np.zeros(3); sqs = np.zeros(3); cnt = np.zeros(3)
    mins = np.full(3, np.inf); maxs = np.full(3, -np.inf)
    res = [[], [], []]
    nan_frac = []
    nframes = 0
    for hha in load_iter:
        if nframes >= n_frames:
            break
        nframes += 1
        nan_frac.append(float(np.mean(~np.isfinite(hha[0]))))
        for c in range(3):
            vals = hha[c].ravel(); vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            sums[c] += vals.sum()
            sqs[c]  += (vals.astype(np.float64) ** 2).sum()
            cnt[c]  += vals.size
            mins[c]  = min(mins[c], float(vals.min()))
            maxs[c]  = max(maxs[c], float(vals.max()))
            if len(res[c]) < 2_000_000:
                step = max(1, vals.size // 4000)
                res[c].extend(vals[::step].tolist())
    return sums, sqs, cnt, mins, maxs, res, nan_frac, nframes

def _iter_scannet():
    for p in _rng15.sample(hha_files, min(SAMPLE_N, len(hha_files))):
        yield torch.load(p, weights_only=True).float().numpy()

def _iter_sun():
    if sun_t is None: return
    for i in _rng15.sample(range(sun_t.shape[0]), min(SAMPLE_N, sun_t.shape[0])):
        yield sun_t[i].float().numpy()

sn = _ch_stats(_iter_scannet(), SAMPLE_N)
su = _ch_stats(_iter_sun(),     SAMPLE_N) if sun_t is not None else None

def _fmt(name, a, b):
    print(f'  {name:<22} {a:>14} {b:>14}')

def _summary(stats):
    sums, sqs, cnt, mins, maxs, res, nan_frac, nframes = stats
    out = {'nframes': nframes, 'nan_frac_mean': float(np.mean(nan_frac)) if nan_frac else float('nan'), 'ch': {}}
    for c, name in enumerate(('disparity_1perm', 'height_m', 'angle_deg')):
        mu = sums[c] / max(cnt[c], 1)
        var = max(0.0, sqs[c] / max(cnt[c], 1) - mu ** 2)
        r = np.array(res[c]) if res[c] else np.array([np.nan])
        d = {
            'mean': float(mu), 'std': float(var ** 0.5),
            'min': float(mins[c]), 'max': float(maxs[c]),
            'p1':  float(np.nanpercentile(r, 1)),
            'p50': float(np.nanpercentile(r, 50)),
            'p99': float(np.nanpercentile(r, 99)),
        }
        if c == 2:
            rr = r[(r >= 0) & (r <= 180)]
            hist, _ = np.histogram(rr, bins=6, range=(0, 180))
            d['hist6_0_180'] = (hist / max(rr.size, 1)).round(3).tolist()
        out['ch'][name] = d
    return out

scn_o = _summary(sn)
sun_o = _summary(su) if su is not None else None

print(f'  {"":<22} {"SUN":>14} {"ScanNet":>14}')
_fmt('frames sampled', sun_o['nframes'] if sun_o else 'n/a', scn_o['nframes'])
_fmt('nan_frac_mean',
     f"{sun_o['nan_frac_mean']:.3f}" if sun_o else 'n/a',
     f"{scn_o['nan_frac_mean']:.3f}")
print('  --- [H1] HEIGHT-CHANNEL FLOOR REFERENCE ---')
_fmt('floor_vis_mean',
     f"{float(np.nanmean(near_floor_su)):.3f}" if near_floor_su is not None else 'n/a',
     f"{float(np.nanmean(near_floor_sn)):.3f}")
_fmt('floor_vis_p10',
     f"{float(np.nanpercentile(near_floor_su, 10)):.3f}" if near_floor_su is not None else 'n/a',
     f"{float(np.nanpercentile(near_floor_sn, 10)):.3f}")
_fmt('height_std (m)',
     f"{sun_o['ch']['height_m']['std']:.3f}" if sun_o else 'n/a',
     f"{scn_o['ch']['height_m']['std']:.3f}")
_fmt('height p1..p99',
     f"{sun_o['ch']['height_m']['p1']:.2f}..{sun_o['ch']['height_m']['p99']:.2f}" if sun_o else 'n/a',
     f"{scn_o['ch']['height_m']['p1']:.2f}..{scn_o['ch']['height_m']['p99']:.2f}")
print('  --- [H2] 8 m DEPTH CLAMP TRUNCATION ---')
# Recompute disp@clamp on the same sample pool (use the eps from §15 H2 block).
def _disp_clamp_frac(load_iter, n_frames, eps=1e-3, clamp_floor=1.0/8.0):
    fracs = []; k = 0
    for hha in load_iter:
        if k >= n_frames: break
        k += 1
        d = hha[0]; dv = d[np.isfinite(d)]
        if dv.size:
            fracs.append(float(np.mean(np.abs(dv - clamp_floor) < eps)))
    return float(np.mean(fracs)) if fracs else float('nan')

scn_clamp = _disp_clamp_frac(_iter_scannet(), SAMPLE_N)
sun_clamp = _disp_clamp_frac(_iter_sun(),     SAMPLE_N) if sun_t is not None else float('nan')
_fmt('disp@clamp frac',
     f"{sun_clamp:.3f}" if not np.isnan(sun_clamp) else 'n/a',
     f"{scn_clamp:.3f}")

print('  --- [H3] ANGLE-CHANNEL SHAPE  (6 bins over [0,180]) ---')
_fmt('angle mean',
     f"{sun_o['ch']['angle_deg']['mean']:.1f}" if sun_o else 'n/a',
     f"{scn_o['ch']['angle_deg']['mean']:.1f}")
print(f"    SUN     hist: {sun_o['ch']['angle_deg']['hist6_0_180'] if sun_o else 'n/a'}")
print(f"    ScanNet hist: {scn_o['ch']['angle_deg']['hist6_0_180']}")

print('\n  READOUT')
print('    H1 fires if ScanNet floor_vis_mean << SUN AND ScanNet height_std/range >> SUN')
print('       -> floor-reference noise; stop trusting per-frame 5th-pct floor on handheld frames.')
print('    H2 fires if ScanNet disp@clamp frac >> SUN -> raise _MAX_DEPTH_M for ScanNet path.')
print('    H3 fires if angle hist differs in shape beyond what RGB mean shift suggests')
print('       -> recheck axis_alignment_rot @ pose_rot on real frames.')

# Record the raw CLI summary into DIAG_RESULTS for downstream comparison.
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
DIAG_RESULTS['hypothesis_panel_cli'] = {
    'scannet': scn_o,
    'sun': sun_o,
    'disp_clamp_frac_mean': {'scannet': scn_clamp, 'sun': sun_clamp},
    'floor_vis': {
        'scannet_mean': float(np.nanmean(near_floor_sn)),
        'sun_mean': float(np.nanmean(near_floor_su)) if near_floor_su is not None else None,
        'scannet_p10': float(np.nanpercentile(near_floor_sn, 10)),
        'sun_p10': float(np.nanpercentile(near_floor_su, 10)) if near_floor_su is not None else None,
    },
}
globals()['DIAG_RESULTS'] = DIAG_RESULTS


## 13. Summary verdict

Single-screen tabulation of every section's verdict so you don't have to mentally aggregate.

In [ ]:
# Tabulate verdicts. Track inconclusive separately from PASS so partial coverage is visible.
DIAG_RESULTS = globals().get('DIAG_RESULTS', {})
# Order: hard-failure-eligible first, then informational/baseline.
ORDER = [
    ('data_leakage',         'Sanity: data leakage'),
    ('floor_angle',          'B: floor-angle convention'),
    ('train_time_nan',       'E: train-time NaN'),
    ('pass2_correlation',    '   Pass 2 correlation'),
    ('extraction_telemetry', 'D: pose-drop fraction'),
    ('drop_list',            'C: drop list per-class'),
    ('float16_roundtrip',    '   float16 round-trip'),
    ('resize_aliasing',      'A: resize aliasing (TV)'),
    ('vs_sun',               '   distribution vs SUN'),
    ('raw_depth_baseline',   '   vs raw-depth baseline'),
    ('h1_floor_visibility',  '15-H1: floor visibility'),
    ('h2_depth_clamp',       '15-H2: 8m depth clamp'),
    ('h3_angle_peaks',       '15-H3: angle peaks vs SUN'),
    ('norm_stats',           '   norm_stats summary'),
]
print('=' * 70)
print('DIAGNOSTIC SUMMARY')
print('=' * 70)
fail_count = warn_count = pass_count = inconclusive_count = skipped_count = 0
for key, label in ORDER:
    r = DIAG_RESULTS.get(key)
    if r is None:
        skipped_count += 1
        print(f'  {label:<35} SKIPPED  (section not run)')
        continue
    v = r.get('verdict', '?')
    detail_keys = {
        'data_leakage':          ('overlap_count',                'overlap={}'),
        'floor_angle':           ('floor_median_deg',             'floor={:.1f}deg'),
        'extraction_telemetry':  ('pose_drop_pct',                'pose-drop={:.1f}%'),
        'train_time_nan':        ('max_post_aug_nan_rate',        'post-aug NaN={:.1%}'),
        'drop_list':             ('max_class_loss_pct',           'worst class loss={:.1f}%'),
        'float16_roundtrip':     ('max_rel_diff',                 'max rel diff={:.4%}'),
        'resize_aliasing':       (None, ''),
        'vs_sun':                ('max_relative_divergence',      'max rel div={:.1%}'),
        'raw_depth_baseline':    ('rgb_max_rel_diff',             'RGB rel diff={:.2%}'),
        'pass2_correlation':     ('worst_rare_median',            'worst rare median={:.4f}'),
        'h1_floor_visibility':   ('scannet_frac_floor_blind_below_5pct', 'floor-blind={:.1%}'),
        'h2_depth_clamp':        ('audit_class_clamp_median',     'large-room clamp={:.1%}'),
        'h3_angle_peaks':        ('scannet_peak_180',             'floor-peak={:.1%}'),
    }
    dk, fmt = detail_keys.get(key, (None, ''))
    detail = fmt.format(r.get(dk, 0)) if dk and dk in r else ''
    marker = ''
    # H1/H2/H3 emit CONFIRMED/PARTIAL/REJECTED; map to FAIL/WARN/PASS for the counter rollup.
    v_norm = {'CONFIRMED': 'FAIL', 'PARTIAL': 'WARN', 'REJECTED': 'PASS'}.get(v, v)
    if v_norm == 'FAIL':
        fail_count += 1; marker = ' <-- ROOT CAUSE CANDIDATE'
    elif v_norm == 'WARN':
        warn_count += 1
    elif v_norm == 'PASS':
        pass_count += 1
    elif v_norm == 'INCONCLUSIVE':
        inconclusive_count += 1
    print(f'  {label:<35} {v:<12}  {detail}{marker}')

print('=' * 70)
parts = []
if fail_count: parts.append(f'{fail_count} FAIL')
if warn_count: parts.append(f'{warn_count} WARN')
if pass_count: parts.append(f'{pass_count} PASS')
if inconclusive_count: parts.append(f'{inconclusive_count} INCONCLUSIVE')
if skipped_count: parts.append(f'{skipped_count} SKIPPED')
print(', '.join(parts))
if fail_count > 0:
    print('Focus on FAIL items first.')
elif warn_count > 0:
    print('No hard failures; combined WARN effects may explain the regression.')
elif inconclusive_count > 0:
    print(f'No FAIL/WARN -- BUT {inconclusive_count} sections inconclusive (missing baseline data).')
    print('Address inconclusives (extract SUN HHA / raw-depth tarball / drop list) before concluding.')
else:
    print('All sections PASS. Regression cause is not in the diagnostics covered here.')

# Persist for later comparison / write-up
_dump_path = '/dev/shm/diag_results.json'
try:
    with open(_dump_path, 'w') as f:
        json.dump(DIAG_RESULTS, f, indent=2, default=str)
    print(f'\nDumped: {_dump_path}')
except OSError as e:
    print(f'Dump failed: {e}')
# Also persist to Drive for post-mortem (survives runtime reset)
import datetime as _dt
_drive_dump = f'/content/drive/MyDrive/datasets/diag_results_{_dt.datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
try:
    os.makedirs(os.path.dirname(_drive_dump), exist_ok=True)
    with open(_drive_dump, 'w') as f:
        json.dump(DIAG_RESULTS, f, indent=2, default=str)
    print(f'Drive backup: {_drive_dump}')
except OSError as e:
    print(f'Drive backup failed: {e}')


## 14. Channel-ablation inference (Tier 2 — decisive cheap test)

Loads your trained HHA checkpoint and computes val MCA in three modes:
1. **Full HHA** — disparity + height + angle (baseline)
2. **Disparity + height** — zero out angle channel only
3. **Disparity only** — zero out angle AND height channels

Tells you directly whether each HHA channel is net-positive or net-negative for your task — without retraining.

**Interpretation:**
- All three within ~1pt: channels are roughly noise-equivalent. Angle/height aren't carrying signal but aren't hurting much either.
- Mode 2 or 3 *higher* than mode 1: the channels are net-negative. Re-extraction with mask-aware downsample probably won't help; HHA is just-not-helpful for this task.
- Mode 1 highest by >2pt: the channels carry useful signal. Then the question is whether they could carry MORE useful signal with better preprocessing — at which point the SUN TV comparison becomes worth running for re-extraction justification.

Best HPO trial (for reference): smoothed val MCA 57.95% with `lr=1.6e-4 wd=2.11e-3 dropout_p=0.32 stem_lr_mult=27`.

In [ ]:
# Fill in the path to your best HHA checkpoint (from the Ray Tune trial dir).
# Common locations: /content/drive/MyDrive/.../<experiment>/<trial>/checkpoint_*/checkpoint.pt
# Or: search for the best by `find <storage_path> -name checkpoint.pt`
CHECKPOINT_PATH = ''  # <-- set this

if not CHECKPOINT_PATH or not os.path.isfile(CHECKPOINT_PATH):
    # Try to auto-locate the best trial
    print('CHECKPOINT_PATH not set or invalid. Searching common locations...')
    _candidates = []
    for _root in ['/content/ray_results', '/content/drive/MyDrive/ray_results',
                  '/content/drive/MyDrive/datasets/ray_results']:
        if os.path.isdir(_root):
            for _path in Path(_root).rglob('checkpoint.pt'):
                _candidates.append(str(_path))
    if _candidates:
        print(f'Found {len(_candidates)} checkpoint(s). First few:')
        for c in _candidates[:5]:
            print(f'  {c}')
        print('\nSet CHECKPOINT_PATH to the best one and re-run this cell.')
    else:
        print('No checkpoints found. Set CHECKPOINT_PATH manually to your trained HHA model.')
else:
    from src.models.linear_integration.li_net3 import li_resnet18
    from src.data_utils.scannet_pretrain_dataset import (
        ScanNetPretrainDataset, _discover_samples,
    )
    from torch.utils.data import DataLoader

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {DEVICE}')

    # Reconstruct model: HHA mode -> stream_input_channels=[3, 3]
    # dropout_p doesn't matter for eval (dropout is off in eval mode)
    model = li_resnet18(
        num_classes=20,
        stream_input_channels=[3, 3],
        dropout_p=0.32,
    ).to(DEVICE)

    # Load checkpoint
    print(f'Loading checkpoint: {CHECKPOINT_PATH}')
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    else:
        sd = ckpt
    # Strip thop buffers if present
    sd = {k: v for k, v in sd.items() if not k.endswith(('.total_ops', '.total_params'))}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:   print(f'  Missing keys (first 5): {missing[:5]}')
    if unexpected:print(f'  Unexpected keys (first 5): {unexpected[:5]}')
    model.eval()

    # Build val dataloader (no aug; deterministic)
    if NS is None:
        raise RuntimeError('norm_stats.json missing -- cannot run inference without it.')
    class_names_path = os.path.join(LOCAL_PATH, 'class_names.txt')
    with open(class_names_path) as f:
        class_names = [l.strip() for l in f if l.strip()]
    val_dir = os.path.join(LOCAL_PATH, 'val')
    val_samples = _discover_samples(val_dir, class_names, use_hha=True)
    val_ds = ScanNetPretrainDataset(
        data_root=LOCAL_PATH, split='val', samples=val_samples,
        class_names=class_names, norm_stats=NS,
        crop_size=224, normalize=True, use_hha=True,
        rgb_aug_prob=0.0, rgb_aug_mag=0.0,
        depth_aug_prob=0.0, depth_aug_mag=0.0,
    )
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=(DEVICE=='cuda'))
    print(f'Val samples: {len(val_ds)}')

    # Compute the post-normalize value that corresponds to (0 - mean) / std
    # for each HHA channel -- that's what "zeroing the channel" means at the
    # input to the depth stream after dataset normalization (since the
    # dataset already normalizes). Equivalent: replace channel with the
    # post-norm value of zero-pre-norm.
    hha_means = torch.tensor(NS['hha_mean'], dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
    hha_stds  = torch.tensor(NS['hha_std'],  dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
    # post-norm value when the raw channel is its mean: (mean - mean)/std = 0
    # so 'zero out a channel' is literally setting the post-norm tensor to 0
    # for that channel index.

    def eval_with_mask(mask_channels):
        """mask_channels: list of channel indices in {0,1,2} to ZERO OUT."""
        model.eval()
        per_class_correct = torch.zeros(20, dtype=torch.long, device=DEVICE)
        per_class_total   = torch.zeros(20, dtype=torch.long, device=DEVICE)
        n_correct = 0; n_total = 0
        with torch.no_grad():
            for rgb, hha, label in tqdm(val_loader, desc=f'mask={mask_channels}'):
                rgb = rgb.to(DEVICE, non_blocking=True)
                hha = hha.to(DEVICE, non_blocking=True)
                label = label.to(DEVICE, non_blocking=True)
                for c in mask_channels:
                    hha[:, c, :, :] = 0.0
                logits = model(rgb, hha)
                pred = logits.argmax(dim=1)
                correct = (pred == label)
                n_correct += int(correct.sum())
                n_total += int(label.numel())
                for cls in range(20):
                    cmask = (label == cls)
                    per_class_total[cls]   += int(cmask.sum())
                    per_class_correct[cls] += int((correct & cmask).sum())
        acc = n_correct / max(n_total, 1)
        per_class_acc = (per_class_correct / per_class_total.clamp(min=1)).cpu().numpy()
        # MCA = mean of per-class accuracies (over classes that have val samples)
        valid = per_class_total.cpu().numpy() > 0
        mca = float(per_class_acc[valid].mean()) if valid.any() else 0.0
        return acc, mca, per_class_acc, per_class_total.cpu().numpy()

    try:
        from tqdm.auto import tqdm
    except ImportError:
        from tqdm import tqdm

    print('\n=== Running 3 inference modes ===\n')
    results = {}
    for label, mask in [
        ('1. full HHA              ', []),
        ('2. disp + height (no ch2)', [2]),
        ('3. disp only (no ch1+ch2)', [1, 2]),
    ]:
        acc, mca, per_class, totals = eval_with_mask(mask)
        results[label] = (acc, mca, per_class, totals)
        print(f'{label}: top1 = {acc*100:.2f}%   MCA = {mca*100:.2f}%')

    print('\n=== Per-class MCA delta ===')
    full_acc = results['1. full HHA              '][2]
    no_angle_acc = results['2. disp + height (no ch2)'][2]
    disp_only_acc = results['3. disp only (no ch1+ch2)'][2]
    print(f"{'class':<28}{'full':>8}{'no-ch2':>10}{'disp-only':>12}")
    print('-' * 60)
    for cls_idx, cn in enumerate(class_names):
        if results['1. full HHA              '][3][cls_idx] == 0:
            continue
        print(f'{cn:<28}{full_acc[cls_idx]*100:>7.1f}%{no_angle_acc[cls_idx]*100:>9.1f}%{disp_only_acc[cls_idx]*100:>11.1f}%')

    # Verdict
    full_mca = results['1. full HHA              '][1]
    no_angle_mca = results['2. disp + height (no ch2)'][1]
    disp_only_mca = results['3. disp only (no ch1+ch2)'][1]
    print('\n=== Verdict ===')
    if disp_only_mca > full_mca + 0.02:
        print(f'CHANNELS NET-NEGATIVE: disp-only beats full by {(disp_only_mca-full_mca)*100:.2f}pt MCA.')
        print('  Angle and/or height channels are hurting on this task.')
        print('  Re-extracting HHA with better preprocessing UNLIKELY to help.')
        print('  Recommend: ship raw-depth pretraining, OR train HHA with channels 1+2 zeroed.')
    elif no_angle_mca > full_mca + 0.02:
        print(f'ANGLE CHANNEL NET-NEGATIVE: no-ch2 beats full by {(no_angle_mca-full_mca)*100:.2f}pt MCA.')
        print('  Height channel helps; angle hurts. Two options:')
        print('  (a) Train with angle channel zeroed -- cheap.')
        print('  (b) Re-extract with mask-aware downsample to clean angle aliasing -- expensive.')
        print('  Run §5 SUN comparison to decide whether re-extraction is justified.')
    elif full_mca > disp_only_mca + 0.02:
        print(f'CHANNELS NET-POSITIVE: full beats disp-only by {(full_mca-disp_only_mca)*100:.2f}pt MCA.')
        print('  HHA channels carry useful signal. The 60->70 regression is NOT explained by channel quality.')
        print('  Look elsewhere -- different optimizer/scheduler, integration weights, etc.')
    else:
        print(f'CHANNELS ROUGHLY NEUTRAL: full={full_mca*100:.2f}, no-ch2={no_angle_mca*100:.2f}, disp-only={disp_only_mca*100:.2f}')
        print('  Angle/height add ~no signal but don\'t hurt. The regression is likely from')
        print('  the HHA encoding adding noise that the model spends capacity ignoring.')

    DIAG_RESULTS['channel_ablation'] = {
        'full_mca': full_mca,
        'no_angle_mca': no_angle_mca,
        'disp_only_mca': disp_only_mca,
        'verdict': ('NET_NEGATIVE' if disp_only_mca > full_mca + 0.02
                    else 'ANGLE_NEG'  if no_angle_mca > full_mca + 0.02
                    else 'NET_POSITIVE' if full_mca > disp_only_mca + 0.02
                    else 'NEUTRAL'),
    }